# ЛР 3 - локальный Qwen

https://colab.research.google.com/drive/14akWrrg-RvLI73Mx2bhy8lVQLhOaAhBF#scrollTo=6XoE33-b8isG

GPU T4. Qwen работает в Colab. Ответы сохраняются на Google Диске в папке `Лаб_3_Квен_локалка`.

In [ ]:
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
FOLDER = Path("/content/drive/MyDrive/Лаб_3_Квен_локалка")
FOLDER.mkdir(exist_ok=True)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto")
llm.eval();

# Запрос и лог

`ask("qwen", prompt, temperature)` возвращает ответ, общее число токенов и время в секундах. JSONL имеет те же шесть полей, что и в VS Code.

In [ ]:
import json
import time

LOG = FOLDER / "answers_qwen.jsonl"

SYSTEM_PROMPT = """Реши предложенную задачу. При необходимости приведи объяснение или код.
Если данных недостаточно для ответа, прямо укажи это.
Последнюю строку оформи строго так:
ОТВЕТ: <краткий итог> | УВЕРЕННОСТЬ: <целое число от 0 до 100>"""

def ask(model, prompt, temperature):
    start = time.perf_counter()
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(llm.device)
    with torch.inference_mode():
        output = llm.generate(
            **inputs, max_new_tokens=2048, do_sample=temperature > 0,
            temperature=temperature or 1.0, top_p=1.0, top_k=0)
    generated = output[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    tokens = inputs["input_ids"].numel() + generated.numel()
    seconds = round(time.perf_counter() - start, 3)
    record = {"model": model, "prompt": prompt, "temperature": temperature,
              "answer": answer, "tokens": tokens, "seconds": seconds}
    with open(LOG, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
    return answer, tokens, seconds

def qwen(prompt):
    return ask("qwen", prompt, 0.2)[0]

In [9]:
answer, tokens, seconds = ask("qwen", "Что такое переобучение? Ответь кратко.", 0)
print(answer)
print(f"Токены: {tokens}, время: {seconds} с")

Переобучение — это ситуация, когда модель слишком хорошо обучается на тренировочных данных и плохо справляется с новыми, неизвестными данными из-за чрезмерной сложности или количества параметров.  

ОТВЕТ: Переобучение — это когда модель слишком хорошо запоминает тренировочные данные и плохо работает на новых данных | УВЕРЕННОСТЬ: 100
Токены: 217, время: 5.53 с


# Отладка промпта

Пять задач из `test_bank.csv` в репозитории, температура 0. 

In [ ]:
from urllib.request import urlretrieve
import csv
import re

urlretrieve("https://raw.githubusercontent.com/SlavaLEVIN/bmstu-ml-lab3/main/test_bank.csv", "test_bank.csv")

with open("test_bank.csv", encoding="utf-8-sig", newline="") as file:
    for task in csv.DictReader(file):
        answer, tokens, seconds = ask("qwen", task["текст_ru"], 0)
        last_line = answer.strip().splitlines()[-1]
        ok = bool(re.fullmatch(r"ОТВЕТ: .+ \| УВЕРЕННОСТЬ: (?:100|[1-9]?[0-9])", last_line))
        print(task["id"], "формат OK" if ok else "формат нарушен")
        print(answer, "\n")

#files.download(str(FOLDER / "answers_qwen.jsonl"))

# Полный банк

Только `questions.csv`, температура 0. Каждый ответ сразу сохраняется. 


In [ ]:
import csv
from pathlib import Path

from urllib.request import urlretrieve

urlretrieve("https://raw.githubusercontent.com/SlavaLEVIN/bmstu-ml-lab3/main/questions.csv", "questions.csv")

def run_bank(model):
    global LOG
    LOG = FOLDER / f"answers_{model}_main.jsonl"
    done = {json.loads(line)["prompt"] for line in LOG.read_text(encoding="utf-8").splitlines()} if LOG.exists() else set()
    with open("questions.csv", encoding="utf-8-sig", newline="") as file:
        for task in csv.DictReader(file):
            if task["текст_ru"] in done:
                continue
            answer, tokens, seconds = ask(model, task["текст_ru"], 0)
            print(task["id"], model, "готово", tokens, "токенов,", seconds, "с")

run_bank("qwen")
files.download(str(FOLDER / "answers_qwen_main.jsonl"))

# Пункт 7. Дополнительный прогон

Один независимый ответ при температуре 0,7 на каждую задачу A, B, D и F02, F03, F05, F09. Всего 22 ответа на модель. Системный промпт и лимит ответа совпадают с основным прогоном.

H3: четыре задачи F с недостатком данных. 

H5: A и B. 

H11: D. Основные гипотезы проверяются по температуре 0.

Лог: `answers_qwen_step7.jsonl`.


In [1]:
import csv

LOG = FOLDER / "answers_qwen_step7.jsonl"
LOG.write_text("", encoding="utf-8")

with open("questions.csv", encoding="utf-8-sig", newline="") as file:
    for task in csv.DictReader(file):
        if task["тип"] in ("A", "B", "D") or task["id"] in ("F02", "F03", "F05", "F09"):
            torch.manual_seed(42)
            answer, tokens, seconds = ask("qwen", task["текст_ru"], 0.7)
            print(task["id"], "готово", seconds, "с", flush=True)

files.download(str(LOG))


NameError: name 'FOLDER' is not defined